# Лабораторная работа № 5

## Конфигурационное пространство и прямая задача кинематики

**Дисциплина:** Введение в технологии виртуальной дополненной реальности и робототехники
**Тема курса:** 6. Конфигурационное пространство и прямая задача кинематики манипулятора

---

**Студент:** _фамилия, имя, группа_
**Дата:**

---

### Цель работы

Освоить два способа описания кинематической цепи — параметры Денавита — Хартенберга
и представление произведением экспонент, — убедиться в их эквивалентности,
и научиться оценивать рабочую зону манипулятора методом Монте-Карло.

### Что понадобится

`numpy`, `sympy`, `matplotlib`.

### Правило практикума

Алгоритм вы реализуете сами. Готовые пакеты робототехники (`roboticstoolbox`,
`spatialmath`) допустимо использовать **только** для сверки — и такую сверку
нужно показать явно.

## Теоретическая справка

### Кинематическая цепь и конфигурационное пространство

Манипулятор — цепь жёстких звеньев, соединённых сочленениями. Каждое вращательное
сочленение добавляет одну степень свободы, и набор всех углов $q = (q_1,\dots,q_n)$
задаёт **конфигурацию**.

Множество всех конфигураций называется конфигурационным пространством. Для цепи
из $n$ вращательных сочленений это тор $T^n = S^1 \times \dots \times S^1$: каждый
угол определён по модулю $2\pi$. Заметьте, что это не $\mathbb{R}^n$ — и по той же
причине, что обсуждалась в лекции 3, тор нельзя гладко и взаимно однозначно
отобразить на $\mathbb{R}^n$.

**Прямая задача кинематики** — это отображение

$$f:\ T^n \to SE(3), \qquad q \mapsto T(q),$$

сопоставляющее конфигурации позу схвата. Оно всегда однозначно и вычисляется явно;
обратное отображение неоднозначно и явной формулы в общем случае не имеет — этим
займёмся в теме 7.

### Параметры Денавита — Хартенберга

Переход между соседними звеньями задаётся четырьмя числами вместо шести: если
оси координат выбраны по специальному правилу, преобразование раскладывается в

$$T_i = R_z(\theta_i)\, T_z(d_i)\, T_x(a_i)\, R_x(\alpha_i).$$

Здесь $a_i$ — длина звена, $\alpha_i$ — угол скрещивания осей, $d_i$ — смещение
вдоль оси, $\theta_i$ — угол поворота. У вращательного сочленения переменная
величина — $\theta_i$, остальные три постоянны.

Полная кинематика — произведение вдоль цепи, ровно как дерево координат из лекции 2:

$$T(q) = T_1(q_1)\, T_2(q_2)\, \dots\, T_n(q_n).$$

### Представление произведением экспонент

Альтернатива, не требующая специального выбора осей. Каждому сочленению
сопоставляется винтовая ось $S_i = (\omega_i,\, v_i)$, где $\omega_i$ — единичный
вектор оси, а $v_i = -\omega_i \times q_i$ для точки $q_i$ на оси. Тогда

$$T(q) = e^{[S_1]q_1}\, e^{[S_2]q_2} \cdots e^{[S_n]q_n} \cdot M,$$

где $M$ — поза схвата в нулевой конфигурации. Матричная экспонента для $se(3)$
вычисляется в замкнутой форме — это прямое обобщение формулы Родрига из лекции 3.

### Рабочая зона и метод Монте-Карло

Рабочая зона — образ конфигурационного пространства при отображении $f$. Для
плоского 2R-манипулятора это кольцо с радиусами $|l_1 - l_2|$ и $l_1 + l_2$,
площадь которого равна $4\pi l_1 l_2$ — редкий случай, когда ответ известен точно
и с ним можно сверить численную оценку.

Метод Монте-Карло даёт погрешность порядка $N^{-1/2}$ независимо от размерности —
именно поэтому в шестимерном случае он остаётся практически единственным способом.

In [35]:
import sys

import numpy as np
import sympy as sp
import time
import matplotlib.pyplot as plt

import colors as c

# промежуток для рандомных чисел векторов сдвига
SHIFT_MAX: int = 5
SHIFT_MIN: int = -SHIFT_MAX
SEED = 0

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(seed=SEED)

# константа как в UE5
SMALL_NUMBER = 1e-12

# ue_is_nearly_zero(val: float) -> bool -- копирка bool IsNearlyZero(float, float) из UE5
def ue_is_nearly_zero(val: float) -> bool:
	return abs(val) <= SMALL_NUMBER

def ue_is_nearly_equal(a: float, b: float) -> bool:
	return abs(a - b) <= SMALL_NUMBER

def ue_is_nearly_pi(val: float) -> bool:
	return abs(val - np.pi) <= 1e-4

def check(expr):
	return f"{c.style(c.bold, c.green)}OK{c.style(c.clear)}" if expr else "{c.style(c.bold, c.red)}FAIL{c.style(c.clear)}"

def normalize(v):
	return v / np.linalg.norm(v)

# параметры UR5: (a, alpha, d) для шести звеньев, метры и радианы
UR5 = [(0.0,      np.pi / 2,  0.089159),
       (-0.425,   0.0,        0.0),
       (-0.39225, 0.0,        0.0),
       (0.0,      np.pi / 2,  0.10915),
       (0.0,     -np.pi / 2,  0.09465),
       (0.0,      0.0,        0.0823)]

#region TASK 1

def R_x(alpha, lib=np):
	c = lib.cos(alpha)
	s = lib.sin(alpha)
	res = [
		[1, 0, 0, 0],
		[0, c, -s, 0],
		[0, s, c, 0],
		[0, 0, 0, 1]
	]
	return sp.Matrix(res) if lib is sp else np.asarray(res, dtype=float)

def R_z(alpha, lib=np):
	c = lib.cos(alpha)
	s = lib.sin(alpha)
	res = [
		[c, -s, 0, 0],
		[s, c, 0, 0],
		[0, 0, 1, 0],
		[0, 0, 0, 1]
	]
	return sp.Matrix(res) if lib is sp else np.asarray(res, dtype=float)

def T_x(a, lib=np):
	res = [
		[1, 0, 0, a],
		[0, 1, 0, 0],
		[0, 0, 1, 0],
		[0, 0, 0, 1]
	]
	return sp.Matrix(res) if lib is sp else np.asarray(res, dtype=float)

def T_z(a, lib=np):
	res = [
		[1, 0, 0, 0],
		[0, 1, 0, 0],
		[0, 0, 1, a],
		[0, 0, 0, 1]
	]
	return sp.Matrix(res) if lib is sp else np.asarray(res, dtype=float)


## Задание 1. Матрица Денавита — Хартенберга

Реализуйте `dh(a, alpha, d, theta, lib)` по формуле из справки. Параметр `lib`
позволит одной и той же функцией считать и численно (`numpy`), и символьно (`sympy`) —
это понадобится в заданиях 2 и 3.

**Контроль.** Для $a = 0{,}5$, $\alpha = 0$, $d = 0$, $\theta = 90°$ схват должен
оказаться в точке $(0;\ 0{,}5;\ 0)$. Проверьте также $\det R = 1$
и $\|R^{\mathsf T}R - I\|$ на уровне машинной погрешности.

**Вопрос для вывода:** почему для перехода между звеньями хватает четырёх чисел,
хотя элемент $SE(3)$ задаётся шестью?

In [36]:

def dh_old(a, alpha, d, theta, lib=np):
	return R_z(theta, lib) @ T_z(d, lib) @ T_x(a, lib) @ R_x(alpha, lib)

def dh(a, alpha, d, theta, lib=np):
	# $$T_i = R_z(\theta_i)\, T_z(d_i)\, T_x(a_i)\, R_x(\alpha_i).$$
	# res = R_z(theta, lib) @ T_z(d, lib) @ T_x(a, lib) @ R_x(alpha, lib)
	ct, st = lib.cos(theta), lib.sin(theta)
	ca, sa = lib.cos(alpha), lib.sin(alpha)
	res = [[ct, -st*ca,  st*sa, a*ct],
			[st,  ct*ca, -ct*sa, a*st],
			[0,   sa,     ca,    d   ],
			[0,   0,      0,     1   ]]
	return sp.Matrix(res) if lib is sp else np.asarray(res, dtype=float)

def __task_1(lib):
	print(f"{c.style(c.yellow)}Используется {c.style(c.green, c.bold)}{lib.__name__}{c.style(c.clear)}")
	T = dh(0.5, 0, 0, lib.pi/2, lib)
	R = T[:3, :3]
	t = T[:3, 3]

	if lib is sp:
		R = sp.Matrix(R)

	p_exp = np.array([0., 0.5, 0.])
	print(f"Ожидаемое значение {p_exp}, получено {t}")
	
	det_func = np.linalg.det if lib is np else sp.det
	det_R = det_func(R)
	print(f"{c.style(c.bold)}R{c.style(c.clear)} = {R}")

	print(f"{c.style(c.bold)}det(R){c.style(c.clear)} = {det_R}\t\t=> {check(ue_is_nearly_equal(det_R, 1))}")
	
	norm_func = np.linalg.norm if lib is np else sp.Matrix.norm
	difference = norm_func(R.T @ R - lib.eye(3))
	print(f"{c.style(c.bold)}||R.T @ R - E||{c.style(c.clear)} = {difference}\t=> {check(ue_is_nearly_zero(difference))}")
	print()

def task_1():
	__task_1(np)
	__task_1(sp)

task_1()

Используется numpy
Ожидаемое значение [0.  0.5 0. ], получено [0.  0.5 0. ]
R = [[ 0. -1.  0.]
 [ 1.  0. -0.]
 [ 0.  0.  1.]]
det(R) = 1.0		=> OK
||R.T @ R - E|| = 0.0	=> OK

Используется sympy
Ожидаемое значение [0.  0.5 0. ], получено Matrix([[0], [0.500000000000000], [0]])
R = Matrix([[0, -1, 0], [1, 0, 0], [0, 0, 1]])
det(R) = 1		=> OK
||R.T @ R - E|| = 0	=> OK



## Задание 2. Плоский 3R символьно

Средствами SymPy перемножьте три матрицы DH с символьными длинами $l_1, l_2, l_3$
и углами $q_1, q_2, q_3$. Извлеките координаты схвата и упростите их.

Сверьте с формулой, которую нетрудно получить вручную из чертежа:

$$x = l_1\cos q_1 + l_2\cos(q_1{+}q_2) + l_3\cos(q_1{+}q_2{+}q_3).$$

Извлеките также ориентацию схвата через `atan2` от элементов матрицы поворота.

**Контроль.** `simplify(x_sym - x_ref) == 0` для обеих координат.

**Вопрос для вывода:** чему равна ориентация схвата и почему ответ выглядит именно
так для плоской цепи?

In [37]:

def task_2():
	l1, l2, l3, q1, q2, q3 = sp.symbols('l1 l2 l3 q1 q2 q3', real=True)

	T = dh(l1, 0, 0, q1, sp) @ dh(l2, 0, 0, q2, sp) @ dh(l3, 0, 0, q3, sp)
	x_sym = sp.simplify(T[0,3])
	y_sym = sp.simplify(T[1,3])
	print(f'{c.style(c.yellow)}Вычислено с помощью SymPy{c.style(c.clear)}')
	print('x =', x_sym)
	print('y =', y_sym)

	# x связан с cos
	# y связан с sin
	x_ref = l1*sp.cos(q1) + l2*sp.cos(q1 + q2) + l3*sp.cos(q1 + q2 + q3)
	y_ref = l1*sp.sin(q1) + l2*sp.sin(q1 + q2) + l3*sp.sin(q1 + q2 + q3)
	
	print(f'\n{c.style(c.yellow)}Референсные формулы из шаблона{c.style(c.clear)}')
	print('x_ref =', x_ref)
	print('y_ref =', y_ref)

	chck_x = sp.simplify(x_sym - x_ref)
	chck_y = sp.simplify(y_sym - y_ref)

	print(f'\n{c.style(c.yellow)}Сверка результатов. Должно равняться нулю{c.style(c.clear)}')
	print(f'x - x_ref = {chck_x}\t=> {check(float(chck_x) == 0)}')
	print(f'y - y_ref = {chck_y}\t=> {check(float(chck_y) == 0)}')

	x_rot = T[0,0] # должно быть cos
	y_rot = T[1,0] # должно быть sin
	orient = sp.atan2(y_rot, x_rot) # atan2(sin(a), cos(a))

	print(f'\n{c.style(c.yellow)}Ориентация схвата{c.style(c.clear)}')
	print(f'x = {sp.simplify(x_rot)}')
	print(f'y = {sp.simplify(y_rot)}')
	print(f'ориентация схвата = {sp.simplify(orient)}')

task_2()

Вычислено с помощью SymPy
x = l1*cos(q1) + l2*cos(q1 + q2) + l3*cos(q1 + q2 + q3)
y = l1*sin(q1) + l2*sin(q1 + q2) + l3*sin(q1 + q2 + q3)

Референсные формулы из шаблона
x_ref = l1*cos(q1) + l2*cos(q1 + q2) + l3*cos(q1 + q2 + q3)
y_ref = l1*sin(q1) + l2*sin(q1 + q2) + l3*sin(q1 + q2 + q3)

Сверка результатов. Должно равняться нулю
x - x_ref = 0	=> OK
y - y_ref = 0	=> OK

Ориентация схвата
x = cos(q1 + q2 + q3)
y = sin(q1 + q2 + q3)
ориентация схвата = atan2(sin(q1 + q2 + q3), cos(q1 + q2 + q3))


## Задание 3. Промышленный манипулятор UR5 и компиляция выражения

В таблице `UR5` заданы реальные параметры шестиосевого манипулятора Universal
Robots UR5. Постройте прямую кинематику двумя путями: численным циклом по матрицам
и символьным произведением с последующей компиляцией через `lambdify`.

Углы скрещивания $\alpha$ задавайте **точными** значениями (`sp.pi/2`), а не числами
с плавающей точкой: тогда синусы и косинусы схлопнутся в $0$ и $\pm1$, и выражение
сократится вдвое.

Сравните время счёта четырёх вариантов: `subs` + `evalf`, `lambdify`,
`lambdify` с исключением общих подвыражений (`cse=True`) и рукописный numpy-цикл.

Затем возьмите 20000 случайных конфигураций, постройте облако положений схвата
и оцените максимальный вылет. Сравните его с оценкой сверху по неравенству
треугольника — суммой всех переносов вдоль цепи.

**Контроль.** Оба способа дают одинаковый результат с точностью $10^{-12}$;
`cse` даёт ускорение не менее чем в 50 раз; измеренный вылет лежит между
$a_2 + a_3$ и оценкой сверху.

**Вопросы для вывода:** во сколько раз ускоряет `cse` и почему так сильно? Обгоняет
ли скомпилированное выражение рукописный numpy-цикл, и если да — почему?

In [38]:
# TODO: задание 3

## Задание 4. Рабочая зона плоского 2R и сходимость Монте-Карло

Для $l_1 = 0{,}5$, $l_2 = 0{,}35$ вычислите точную площадь кольца.
Затем оцените её методом Монте-Карло: бросайте точки в квадрат со стороной
$2(l_1{+}l_2)$ и считайте долю попавших в кольцо.

Для $N$ от $10^2$ до $10^6$ усредните ошибку по 30 независимым запускам, постройте
график в двойном логарифмическом масштабе и оцените наклон.

**Контроль.** Наклон должен получиться близким к $-0{,}5$.

**Вопросы для вывода:** почему точная площадь равна $4\pi l_1 l_2$? Во сколько раз
надо увеличить $N$, чтобы уменьшить ошибку вдвое? Зависит ли скорость сходимости
от размерности задачи?

In [39]:
# TODO: задание 4

## Задание 5. Объём рабочей зоны UR5

Для шестиосевого манипулятора точной формулы нет. Оцените объём рабочей зоны
по облаку точек: разбейте пространство на воксели со стороной $h$ и умножьте число
занятых вокселей на $h^3$.

Посчитайте оценку при $h = 0{,}08$, $0{,}05$, $0{,}03$ м, а затем при фиксированном
$h = 0{,}05$ проследите, как оценка меняется с ростом числа выборок.

**Контроль.** Оценка должна расти с числом выборок.

**Вопросы для вывода:** почему оценка растёт с $N$ и не выходит на плато даже
при 20000 выборок? Почему она так сильно зависит от $h$? В какую сторону смещена
такая оценка и почему? Сколько примерно выборок нужно, чтобы заполнить все воксели
при $h = 0{,}05$?

In [40]:
# TODO: задание 5

## Задание 6. Произведение экспонент и его эквивалентность DH

Реализуйте `exp_se3(S, theta)` — матричную экспоненту для винтовой оси.
Формула прямо обобщает Родрига из лекции 3:

$$e^{[S]\theta} = \begin{pmatrix} e^{[\omega]\theta} & G(\theta)\,v \\ 0 & 1\end{pmatrix},
\quad G(\theta) = I\theta + (1-\cos\theta)[\omega] + (\theta - \sin\theta)[\omega]^2 .$$

Задайте для плоского 3R матрицу $M$ нулевой конфигурации и три винтовые оси,
после чего сравните результат с DH-версией на 500 случайных конфигурациях.

**Контроль.** Расхождение на уровне машинной погрешности ($\sim 10^{-16}$).

**Вопрос для вывода:** в чём преимущество представления произведением экспонент
перед DH, если результат тот же?

In [41]:
# TODO: задание 6

## Задание 7. Движение по последовательности конфигураций

Постройте функцию `arm_points(q)`, возвращающую координаты всех сочленений
плоского 3R. Задайте линейную интерполяцию между двумя конфигурациями по 60 шагам
и нарисуйте несколько промежуточных поз с нарастающей непрозрачностью, а поверх —
траекторию схвата.

**Вопрос для вывода:** траектория схвата при линейной интерполяции в конфигурационном
пространстве — прямая линия в рабочем пространстве? Почему?

In [42]:
# TODO: задание 7

## Выводы

_Ответьте на все вопросы, отмеченные как «вопрос для вывода», приводя конкретные числа._

## Контрольные вопросы

1. Что такое конфигурационное пространство? Почему для цепи из $n$ вращательных
   сочленений это тор $T^n$, а не $\mathbb{R}^n$?
2. Почему прямая задача кинематики всегда разрешима однозначно, а обратная — нет?
3. Какой геометрический смысл у каждого из четырёх параметров DH?
4. Почему для перехода между соседними звеньями хватает четырёх чисел вместо шести?
5. В чём недостатки соглашения DH и что предлагает взамен представление
   произведением экспонент?
6. Запишите формулу винтовой оси $S = (\omega, v)$ для вращательного сочленения.
   Каков геометрический смысл вектора $v$?
7. Как связаны формула Родрига из лекции 3 и матричная экспонента для $se(3)$?
8. Что такое рабочая зона? Чему равна её площадь для плоского 2R-манипулятора?
9. Почему погрешность метода Монте-Карло убывает как $N^{-1/2}$ и не зависит
   от размерности?
10. Почему воксельная оценка объёма смещена вниз и как это исправить?